***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [8. 校准](8_0_introduction.ipynb)
    * 上一节： [8.1 作为最小二乘问题的校准](8_1_calibration_least_squares_problem.ipynb)
    * 下一节： [8.3 第二代校准（2GC）](8_3_2gc.ipynb)

***


导入标准模块:


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    from IPython.display import HTML, display
except ImportError:
    HTML = None
    display = None

STYLE_PATH = Path("../style/course.css")
TOGGLE_PATH = Path("../style/code_toggle.html")

if HTML is not None and display is not None:
    if STYLE_PATH.exists():
        display(HTML(f"<style>{STYLE_PATH.read_text(encoding='utf-8')}</style>"))
    if TOGGLE_PATH.exists():
        display(HTML(TOGGLE_PATH.read_text(encoding="utf-8")))

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=3, suppress=True)

RNG = np.random.default_rng(20260419)


第一代校准（1GC）解决的是最经典、也最普遍的一类问题：我们利用一个或多个外场校准源去追踪系统增益、带通、延迟或绝对通量标尺，然后把这些解转移到目标场。

它的成功建立在一个关键假设上：在校准源方向上求得的误差项，能够在目标场方向上近似成立。这个假设在窄视场、较稳定的大气和方向依赖效应不太严重时通常是合理的，因此 1GC 仍然是大多数处理链的起点。


***


## 8.2 第一代校准（1GC）<a id='cal:sec:1gccal'></a>

在现代软件中，1GC 往往表现为一条看起来很“工程化”的解链，例如 `setjy -> bandpass -> gaincal -> fluxscale -> applycal`。但在算法上，它仍然是一个把观测数据、校准源模型和天线增益联系起来的求解问题。

这一节重点关注三件事：

- 校准源为什么能约束系统增益；
- 为什么解需要在时间和频率上插值、平滑或转移；
- 为什么闭合量是诊断 1GC 是否合理的重要工具。


In [ ]:
def baseline_pairs(nant):
    return [(p, q) for p in range(nant) for q in range(p + 1, nant)]


def point_source_model_cube(flux, nt, nant):
    model = np.full((nt, nant, nant), flux + 0.0j, dtype=complex)
    for p in range(nant):
        model[:, p, p] = 0.0
    return model


def apply_gains(model, gains, noise_std=0.0, rng=None):
    data = gains[:, :, None] * model * np.conj(gains[:, None, :])
    if noise_std > 0.0:
        if rng is None:
            rng = np.random.default_rng(0)
        noise = noise_std * (
            rng.normal(size=data.shape) + 1j * rng.normal(size=data.shape)
        )
        data = data + noise
    for p in range(data.shape[1]):
        data[:, p, p] = 0.0
    return data


def solve_gains(data, model, n_iter=30, ref_ant=0, phase_only=False):
    nt, nant, _ = data.shape
    gains = np.ones((nt, nant), dtype=complex)
    eps = 1e-12

    for t in range(nt):
        gt = np.ones(nant, dtype=complex)
        for _ in range(n_iter):
            new = gt.copy()
            for p in range(nant):
                mask = np.ones(nant, dtype=bool)
                mask[p] = False
                num = np.sum(data[t, p, mask] * gt[mask] * np.conj(model[t, p, mask]))
                den = np.sum(
                    np.abs(gt[mask]) ** 2 * np.abs(model[t, p, mask]) ** 2
                ) + eps
                new[p] = num / den

            if phase_only:
                amp = np.maximum(np.abs(new), eps)
                new = new / amp

            ref = new[ref_ant]
            new = new / (ref / max(np.abs(ref), eps))
            gt = new

        gains[t] = gt

    return gains


def gain_track(times, nant):
    ant_phase = np.linspace(0.0, np.pi, nant, endpoint=False)
    amp = 1.0 + 0.04 * np.sin(times[:, None] * (0.8 + 0.05 * np.arange(nant)) + ant_phase)
    phase = 0.25 * np.sin(times[:, None] * (1.2 + 0.08 * np.arange(nant)) + 0.5 * ant_phase)
    return amp * np.exp(1j * phase)


def interpolate_complex(times_src, gains_src, times_dst):
    amp = np.abs(gains_src)
    phase = np.unwrap(np.angle(gains_src))
    amp_i = np.vstack(
        [np.interp(times_dst, times_src, amp[:, ant]) for ant in range(amp.shape[1])]
    ).T
    phase_i = np.vstack(
        [np.interp(times_dst, times_src, phase[:, ant]) for ant in range(phase.shape[1])]
    ).T
    return amp_i * np.exp(1j * phase_i)


def closure_phase(vis12, vis23, vis31):
    return np.angle(vis12 * vis23 * vis31)


def closure_amplitude(vis12, vis34, vis13, vis24):
    return np.abs(vis12 * vis34 / (vis13 * vis24))


### 8.2.1 校准源观测与解转移 <a id='cal:sec:calobs'></a>

典型的 1GC 观测不会只看目标源，而是把几类校准源穿插进时间轴中：

- **绝对通量校准源**：定义整体振幅标尺；
- **带通校准源**：追踪频率响应；
- **相位校准源**：高时间频率地追踪大气和电子链路相位变化；
- **目标场**：真正的科学观测。

下面这张示意图把这种“交替观测”的节奏表现出来。真正的阵列调度更复杂，但核心思想相同：用较短、较可靠的校准扫描去支撑更长的目标场积分。


In [ ]:
blocks = [
    ("通量校准源", "tab:orange", [(0.0, 0.35), (5.5, 5.85)]),
    ("相位校准源", "tab:green", [(0.8, 1.0), (2.1, 2.3), (3.4, 3.6), (4.7, 4.9)]),
    ("目标场", "tab:blue", [(0.35, 0.8), (1.0, 2.1), (2.3, 3.4), (3.6, 4.7), (4.9, 5.5)]),
]

fig, ax = plt.subplots(figsize=(10, 2.8))

for idx, (label, color, spans) in enumerate(blocks):
    for start, stop in spans:
        ax.barh(idx, stop - start, left=start, height=0.45, color=color, alpha=0.85)

ax.set_yticks(range(len(blocks)))
ax.set_yticklabels([item[0] for item in blocks])
ax.set_xlabel("观测时间 [hour]")
ax.set_title("1GC 中常见的交替观测节奏")
ax.set_xlim(0.0, 6.0)
plt.tight_layout()


这个时间结构直接决定了解的物理含义。若相位校准源与目标场相距很远，或者扫描间隔太长，那么即便校准源上的解很好，也可能无法准确代表目标场上的传播误差。后面我们会用一个简单模拟来量化这种“转移误差”。


### 8.2.2 在点源校准源上求解方向无关增益

对位于相位中心、结构已知的点源，模型可见度矩阵最简单。下面的例子用一个亮点源模拟相位校准源，再用交替迭代求解器恢复每面天线的复增益。你可以把它视为 `gaincal` 一类任务背后的最简化原型。


In [ ]:
nant = 6
times_cal = np.linspace(0.0, 6.0, 36)
model_cal = point_source_model_cube(flux=5.0, nt=times_cal.size, nant=nant)

true_gains_cal = gain_track(times_cal, nant)
data_cal = apply_gains(model_cal, true_gains_cal, noise_std=0.05, rng=RNG)
solved_gains_cal = solve_gains(data_cal, model_cal, n_iter=35, ref_ant=0, phase_only=False)

baseline = (1, 4)
corrected_cal = data_cal / (
    solved_gains_cal[:, :, None] * np.conj(solved_gains_cal[:, None, :]) + 1e-12
)

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex="col")

axes[0, 0].plot(times_cal, np.abs(true_gains_cal[:, 3]), color="black", lw=2.0, label="真实")
axes[0, 0].plot(times_cal, np.abs(solved_gains_cal[:, 3]), color="tab:blue", lw=1.8, label="求解")
axes[0, 0].set_ylabel("|g_3|")
axes[0, 0].legend(loc="upper right")
axes[0, 0].set_title("Antenna 3 amplitude")

axes[1, 0].plot(times_cal, np.angle(true_gains_cal[:, 3]), color="black", lw=2.0, label="真实")
axes[1, 0].plot(times_cal, np.angle(solved_gains_cal[:, 3]), color="tab:blue", lw=1.8, label="求解")
axes[1, 0].set_ylabel("phase [rad]")
axes[1, 0].set_xlabel("time [hour]")
axes[1, 0].set_title("Antenna 3 phase")

axes[0, 1].plot(times_cal, np.abs(data_cal[:, baseline[0], baseline[1]]), color="tab:red", lw=1.5)
axes[0, 1].plot(
    times_cal,
    np.abs(corrected_cal[:, baseline[0], baseline[1]]),
    color="tab:blue",
    lw=1.8,
)
axes[0, 1].axhline(5.0, color="black", ls="--")
axes[0, 1].set_ylabel("amplitude [Jy]")
axes[0, 1].set_title(f"Baseline {baseline[0]}-{baseline[1]} amplitude")

axes[1, 1].plot(
    times_cal,
    np.angle(data_cal[:, baseline[0], baseline[1]] / 5.0),
    color="tab:red",
    lw=1.5,
    label="改正前",
)
axes[1, 1].plot(
    times_cal,
    np.angle(corrected_cal[:, baseline[0], baseline[1]] / 5.0),
    color="tab:blue",
    lw=1.8,
    label="改正后",
)
axes[1, 1].axhline(0.0, color="black", ls="--")
axes[1, 1].set_ylabel("phase error [rad]")
axes[1, 1].set_xlabel("time [hour]")
axes[1, 1].set_title(f"Baseline {baseline[0]}-{baseline[1]} phase")
axes[1, 1].legend(loc="upper right")

plt.tight_layout()


可以看到，求解器能够把“基线上的复误差”分解回“天线上的复增益”。这正是天线基校准的核心优势。只要污染主要来自每面天线自身的方向无关误差，基线数目随 $N_{\rm ant}^2$ 增长，而待求参数只随 $N_{\rm ant}$ 增长，因此问题通常是强约束的。


### 8.2.3 解转移、插值与目标场剩余误差

下面把同一组校准解转移到目标场。为了模拟“校准源和目标场不完全等价”，我们额外加入一个与方向相关的相位残差项，代表校准源与目标场穿过不同大气路径后的差异。你会看到，1GC 虽然显著改善了数据，但不能把这类误差全部消除。


In [ ]:
times_target = np.linspace(0.15, 5.95, 96)
model_target = point_source_model_cube(flux=1.5, nt=times_target.size, nant=nant)

common_gains = gain_track(times_target, nant)
ant_phase = np.linspace(0.0, np.pi, nant, endpoint=False)
target_extra_phase = 0.12 * np.sin(2.8 * times_target[:, None] + ant_phase[None, :])
target_direction_term = np.exp(1j * target_extra_phase)
true_gains_target = common_gains * target_direction_term

data_target = apply_gains(model_target, true_gains_target, noise_std=0.02, rng=RNG)

interp_solutions = interpolate_complex(times_cal, solved_gains_cal, times_target)
corrected_target = data_target / (
    interp_solutions[:, :, None] * np.conj(interp_solutions[:, None, :]) + 1e-12
)

test_bl = (0, 5)
raw_phase = np.angle(data_target[:, test_bl[0], test_bl[1]] / 1.5)
corr_phase = np.angle(corrected_target[:, test_bl[0], test_bl[1]] / 1.5)

rms_raw = np.sqrt(np.mean(raw_phase**2))
rms_corr = np.sqrt(np.mean(corr_phase**2))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

axes[0].plot(times_target, raw_phase, color="tab:red", lw=1.5, label="改正前")
axes[0].plot(times_target, corr_phase, color="tab:blue", lw=1.8, label="1GC 后")
axes[0].axhline(0.0, color="black", ls="--")
axes[0].set_xlabel("time [hour]")
axes[0].set_ylabel("baseline phase error [rad]")
axes[0].set_title(f"Baseline {test_bl[0]}-{test_bl[1]} target phase")
axes[0].legend(loc="upper right")

axes[1].plot(times_target, np.abs(data_target[:, test_bl[0], test_bl[1]]), color="tab:red", lw=1.5)
axes[1].plot(
    times_target,
    np.abs(corrected_target[:, test_bl[0], test_bl[1]]),
    color="tab:blue",
    lw=1.8,
)
axes[1].axhline(1.5, color="black", ls="--")
axes[1].set_xlabel("time [hour]")
axes[1].set_ylabel("amplitude [Jy]")
axes[1].set_title("Transferred solutions improve, but do not fully remove, target errors")

plt.tight_layout()
print(f"目标场相位 RMS：改正前 {rms_raw:.3f} rad，1GC 后 {rms_corr:.3f} rad")


这正是实践中常见的情形：1GC 通常能去掉最大尺度、最平滑的系统误差，但目标场仍会留下模型不匹配或方向依赖残差。也因此，1GC 之后往往还要进入自校准，甚至进一步进入 3GC。


### 8.2.4 闭合量：为什么它们是诊断工具 <a id='cal:sec:closure'></a>

闭合相位和闭合振幅是 1GC 中最重要的诊断量之一。对纯天线基的方向无关增益来说，闭合量会把每面天线自身的复增益抵消掉，因此它们反映的是**天空结构与非天线基误差**，而不是简单的增益漂移。


In [ ]:
u12, u23, u31 = 42.0, 31.0, -73.0
u13, u24, u34 = 73.0, 55.0, 24.0

fluxes = np.array([1.0, 0.35, 0.18])
ls = np.array([0.0, 0.012, -0.019])

def structured_vis(u):
    return np.sum(fluxes * np.exp(-2j * np.pi * u * ls))


v12 = structured_vis(u12)
v23 = structured_vis(u23)
v31 = structured_vis(u31)
v13 = structured_vis(u13)
v24 = structured_vis(u24)
v34 = structured_vis(u34)

g = np.array(
    [
        1.05 * np.exp(1j * 0.35),
        0.92 * np.exp(-1j * 0.18),
        1.08 * np.exp(1j * 0.12),
        0.97 * np.exp(-1j * 0.25),
    ]
)

v12_obs = g[0] * np.conj(g[1]) * v12
v23_obs = g[1] * np.conj(g[2]) * v23
v31_obs = g[2] * np.conj(g[0]) * v31
v13_obs = g[0] * np.conj(g[2]) * v13
v24_obs = g[1] * np.conj(g[3]) * v24
v34_obs = g[2] * np.conj(g[3]) * v34

cp_true = closure_phase(v12, v23, v31)
cp_obs = closure_phase(v12_obs, v23_obs, v31_obs)
ca_true = closure_amplitude(v12, v34, v13, v24)
ca_obs = closure_amplitude(v12_obs, v34_obs, v13_obs, v24_obs)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].bar(["真实闭合相位", "受增益污染后"], [cp_true, cp_obs], color=["black", "tab:blue"])
axes[0].set_ylabel("closure phase [rad]")
axes[0].set_title("Closure phase is gain-invariant")

axes[1].bar(["真实闭合振幅", "受增益污染后"], [ca_true, ca_obs], color=["black", "tab:green"])
axes[1].set_ylabel("closure amplitude")
axes[1].set_title("Closure amplitude is also gain-invariant")
plt.tight_layout()

print(f"闭合相位：真实 {cp_true:.4f} rad，受污染后 {cp_obs:.4f} rad")
print(f"闭合振幅：真实 {ca_true:.4f}，受污染后 {ca_obs:.4f}")


这也是为什么在真实处理时，我们经常先看闭合量，再看求解器是否“收敛”。如果闭合量本身已经表现出强烈的非物理结构，那么问题可能不是一个简单的天线基增益，而是 RFI、相关器异常、错误的天空模型，或更复杂的方向依赖误差。


### 8.2.5 1GC 的实践解链

对连续谱观测，一个典型的 1GC 处理链通常包含下列步骤：

- `listobs` / `plotms`：先做数据检查，确认观测结构和异常扫描；
- `setjy`：给绝对通量校准源设定模型；
- `bandpass`：求带通响应；
- `gaincal`：求时间依赖增益与相位；
- `fluxscale`：把振幅解绑定到绝对通量标尺；
- `applycal`：把解转移到目标场；
- `plotms` / quick image：检查改正后数据与图像质量。

真正的关键不在命令名字本身，而在于理解每一步对应什么物理量、它假设了什么、以及失败时最先会在哪类诊断图上暴露出来。
